OBS: No Roboflow ao misturar Retangulos e Poligonos, se exportar como Yolov8 pode apresentar erros, por isso farei manualmente a conversão de COCOMM para Yolo


# 1. Imports

In [2]:
import os
import json
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import Rectangle
import sys
import torch
import torchvision
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import CocoDetection
from torch.cuda.amp import autocast, GradScaler
from datetime import datetime
#import onnx
#import onnxruntime as ort
from PIL import Image
import numpy as np
from torch.optim.lr_scheduler import StepLR
import time


base_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
print(base_dir)

sys.path.append(base_dir)

%matplotlib inline

d:\Arquivos\ProjetosPython\PICOS


In [3]:
torch.cuda.is_available() 

True

In [4]:
import os
from ultralytics import RTDETR
import cv2
import matplotlib.pyplot as plt
import torch

# 1. Configurações Iniciais
device = "cuda" if torch.cuda.is_available() else "cpu"

def create_and_train_rtdetr(data_yaml_path, epochs=100):
    """
    Diferente do Faster R-CNN, o RT-DETR (Ultralytics) gerencia o loop
    de treino internamente, o que é muito mais estável.
    O data_yaml_path deve apontar para um arquivo .yaml no formato YOLO/COCO.
    """
    # Carrega o modelo 'Large' (mais parrudo que a ResNet50)
    # Licença: Apache 2.0
    model = RTDETR("rtdetr-l.pt") 

    # Inicia o treino
    model.train(
        data=data_yaml_path,
        epochs=epochs,
        imgsz=640,
        batch=4,
        device=device,
        project="projeto_biscoito",
        name="treino_rtdetr"
    )
    return model

def load_rtdetr_eval(model_path):
    """Carrega o modelo treinado para avaliação"""
    model = RTDETR(model_path)
    return model

def visualize_predictions_rtdetr(model, image_path, threshold=0.5):
    """
    Equivalente à sua visualize_predictions_image, mas otimizada.
    O RT-DETR nativamente não requer NMS manual.
    """
    # Realiza a predição
    results = model.predict(source=image_path, conf=threshold, device=device)[0]

    # Converte imagem para exibição
    image_rgb = cv2.cvtColor(results.orig_img, cv2.COLOR_BGR2RGB)
    
    # Extrai dados das detecções
    boxes = results.boxes.xyxy.cpu().numpy()  # [x1, y1, x2, y2]
    scores = results.boxes.conf.cpu().numpy()
    
    fig, ax = plt.subplots(1, 2, figsize=(16, 8))

    # Subplot 1: Original
    ax[0].imshow(image_rgb)
    ax[0].axis('off')
    ax[0].set_title('Imagem Original')

    # Subplot 2: Detecções (RT-DETR raramente duplica aqui)
    ax[1].imshow(image_rgb)
    for box, score in zip(boxes, scores):
        x1, y1, x2, y2 = box
        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1,
                             linewidth=2, edgecolor='r', facecolor='none')
        ax[1].add_patch(rect)
        ax[1].text(x1, y1, f'{score:.2f}', color='white', fontsize=8, 
                   bbox=dict(facecolor='red', alpha=0.5))

    ax[1].set_title(f'RT-DETR: {len(boxes)} Biscoitos')
    ax[1].axis('off')
    plt.show()

def resume_train_rtdetr(data, last_weights_path, epochs=100):
    # 1. Carrega o modelo a partir do último checkpoint
    model = RTDETR(last_weights_path)

    # 2. Inicia o treino corrigindo a instabilidade da GTX 1660
    model.train(
        data=data,
        epochs=epochs,
        imgsz=640,
        batch=4,              # Reduzido para 2 para não estourar os 6GB de VRAM
        amp=False,            # OBRIGATÓRIO: Resolve o problema do mAP 0 na GTX 16xx
        deterministic=False,  # Resolve o erro de grid_sampler_2d
        resume=False,         # Mudamos para False para resetar o otimizador bugado
        device=device
    )
    return model



In [ ]:
# Exemplo de uso:
import torch
import gc

torch.cuda.empty_cache()
gc.collect()
model = create_and_train_rtdetr(r"D:\Arquivos\ProjetosPython\PICOS\data\inputs\train_images\YOLOV8_20260105\data.yaml")

Ultralytics 8.3.248  Python-3.8.10 torch-2.4.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1660, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\Arquivos\ProjetosPython\PICOS\data\inputs\train_images\YOLOV8_20260105\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=treino_rtdetr7, nbs=64, nms=False, opset=None, optimize=False, optimizer=

d:\Arquivos\ProjetosPython\PICOS\.venv\lib\site-packages\torch\autograd\graph.py:769: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\Context.cpp:87.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      1/100      5.46G      0.791     0.5206     0.3361        106        640: 100% ━━━━━━━━━━━━ 117/117 9.3s/it 18:048.9ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.3it/s 4.5s0.7ss
                   all         43        989      0.842      0.837      0.874       0.58

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


d:\Arquivos\ProjetosPython\PICOS\.venv\lib\site-packages\torch\autograd\graph.py:769: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\Context.cpp:87.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      2/100      5.48G     0.4359     0.4709     0.1132        191        640: 100% ━━━━━━━━━━━━ 117/117 3.4s/it 6:373.4ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 2.2it/s 2.8s0.6ss
                   all         43        989      0.892      0.893      0.927      0.654

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


...

     16/100      5.33G     0.3476     0.4103    0.08433        158        640: 100% ━━━━━━━━━━━━ 117/117 18.5s/it 36:0125.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.4s/it 8.3s0.6s3
                   all         43        989       0.95      0.943      0.956      0.689

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


d:\Arquivos\ProjetosPython\PICOS\.venv\lib\site-packages\torch\autograd\graph.py:769: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\Context.cpp:87.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     17/100      5.49G     0.3532     0.4047    0.08318        115        640: 100% ━━━━━━━━━━━━ 117/117 15.4s/it 30:0317.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.7it/s 3.5s0.7ss
                   all         43        989      0.966      0.964      0.964      0.679

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


d:\Arquivos\ProjetosPython\PICOS\.venv\lib\site-packages\torch\autograd\graph.py:769: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\Context.cpp:87.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     18/100      5.49G     0.3532     0.4141    0.07421        143        640: 17% ━━────────── 20/117 31.3s/it 8:22<50:413

In [ ]:
# Exemplo de uso:
import torch
import gc

torch.cuda.empty_cache()
gc.collect()

# Se você quiser carregar o modelo que já foi treinado:
data = r"D:\Arquivos\ProjetosPython\PICOS\data\inputs\train_images\YOLOV8_20260105\data.yaml"
model_path = r"D:\Arquivos\ProjetosPython\PICOS\pipeline\projeto_biscoito\treino_rtdetr7\weights\best.pt" 

# Reinicia do ponto que travou
model = resume_train_rtdetr(data, model_path, epochs=100)

Ultralytics 8.3.248  Python-3.8.10 torch-2.4.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1660, 6144MiB)
engine\trainer: agnostic_nms=False, amp=False, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\Arquivos\ProjetosPython\PICOS\data\inputs\train_images\YOLOV8_20260105\data.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=D:\Arquivos\ProjetosPython\PICOS\pipeline\projeto_biscoito\treino_rtdetr7\weights\best.pt, momentum=0.937, mosaic=1.0, multi_scale=False